In [1]:
from bs4 import BeautifulSoup
import json

In [2]:
def read_cv(path_cv):
    with open(path_cv, "r", encoding="utf-8") as f:
        soup = BeautifulSoup(f, "html.parser")
    return soup

path_html = 'data/curriculos/2747150211073176/cv.html'
soup = read_cv(path_html)

In [3]:
sections = [
  "Artigos completos publicados em periódicos",
  "Livros publicados/organizados ou edições",
  "Capítulos de livros publicados",
  "Textos em jornais de notícias/revistas",
  "Trabalhos completos publicados em anais de congressos",
  "Resumos expandidos publicados em anais de congressos",
  "Resumos publicados em anais de congressos",
  "Apresentações de Trabalho",
  "Outras produções bibliográficas"
  ]

In [4]:
producoes = {}
for section_header in soup.select('div.cita-artigos b'):
    section_name = section_header.get_text(strip=True)
    if section_name in sections:
        items = []
        parent = section_header.find_parent('div', class_='cita-artigos')
        sibling = parent.find_next_sibling()
        
        while sibling and 'cita-artigos' not in sibling.get('class', []):
            for span in sibling.select('span.transform'):
                items.append(span)
            sibling = sibling.find_next_sibling()
        if items:
            producoes[section_name] = items

# Artigos completos

In [ ]:
list_artigos = producoes["Artigos completos publicados em periódicos"]

In [ ]:
from lib.parser.lattes.artigos_completos import slipt_artigos


c_doi, s_doi = slipt_artigos(list_artigos)

In [ ]:
livros = producoes["Livros publicados/organizados ou edições"]

In [5]:
capitulos = producoes["Capítulos de livros publicados"]
len(capitulos)

70

In [6]:
from lib.parser.lattes.capitulo_livros import parser_capitulos



refs = parser_capitulos(capitulos)

In [ ]:
from datetime import date


def normalize_capitulo(capitulo):
    publication = {
        'publication_type': 'chapter-book',
        'title': capitulo.get('titulo-do-capitulo'),
        'date_published': date(int(capitulo.get('ano')), 1, 1),
        'doi': capitulo.get('doi'),
        'publisher': capitulo.get('editora'),
        'volume_number': capitulo.get('volume'),
        'page_start': capitulo.get('page_start'),
        'page_end': capitulo.get('page_end'),
        'edition': capitulo.get('edicao'),
        'source': 'lattes'        
    }
    return publication
    
for ref in refs:
    publication = normalize_capitulo(ref)
    print(publication)

In [15]:
id_lattes = '2747150211073176'  
with open(f'data/curriculos/{id_lattes}/capitulos.jsonl', 'w', encoding='utf-8') as f:
    for capitulo in refs:
        json.dump(capitulo, f, ensure_ascii=False)
        f.write('\n')

# Textos em jornais de notícias/revistas

In [8]:
text_news = producoes["Textos em jornais de notícias/revistas"]

In [ ]:
from lib.parser.lattes.text_news import normalize_news


refs = []
for i in text_news:
    norm_groq = normalize_news(i.text)
    refs.append(norm_groq)
    print(norm_groq)

In [26]:
id_lattes = '2747150211073176'  
with open(f'data/curriculos/{id_lattes}/text_news.jsonl', 'w', encoding='utf-8') as f:
    for capitulo in refs:
        json.dump(refs, f, ensure_ascii=False)
        f.write('\n')

In [28]:
from lib.parser.lattes.text_news import normalize_text_news






ImportError: cannot import name 'normalize_text_news' from 'lib.parser.lattes.text_news' (/home/inacio/orbis/lib/parser/lattes/text_news.py)

In [ ]:
for ref in refs:
    d = normalize_text_news(ref)
    print(d)
    
    

{'autores': ['VAL, A. L.', 'MACHADO, J. A. C.'],
 'titulo': 'Desafios amazônicos presentes e futuros',
 'titulo-da-revista': 'Boa Vontade',
 'cidade': None,
 'data': None}